In [1]:
import torch, torchvision
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
import os, random, time, json, math
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

SEED = 3367
HASH_BITS = 32            # thử 16, 32, 48, 64; mỗi lần chạy một giá trị
BATCH_SIZE = 64
EPOCHS_PER_BETA = 20      # 1–2 chỉ dùng để smoke-test, không phải thí nghiệm cuối
BETA_SCHEDULE = [1, 2, 4, 8, 16]
LR = 1e-5
NUM_WORKERS = 2
DATA_ROOT = "./data"
OUTPUT_DIR = "./hashnet_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


3. Tạo split CIFAR-10 theo giao thức 5k / 1k / 54k

In [3]:
# AlexNet pretrained cần kích thước đầu vào phù hợp; dùng ImageNet normalization.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Dùng hai bản dataset để train có augmentation nhưng inference không augmentation.
train_aug = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=train_transform)
train_eval = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=eval_transform)
test_eval = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=eval_transform)

y_train = np.array(train_aug.targets)
y_test = np.array(test_eval.targets)
rng = np.random.default_rng(SEED)

train_idx = []
for c in range(10):
    idx_c = np.where(y_train == c)[0]
    train_idx.extend(rng.choice(idx_c, size=500, replace=False).tolist())
train_idx = np.array(sorted(train_idx), dtype=np.int64)

query_idx = []
for c in range(10):
    idx_c = np.where(y_test == c)[0]
    query_idx.extend(rng.choice(idx_c, size=100, replace=False).tolist())
query_idx = np.array(sorted(query_idx), dtype=np.int64)

train_idx_set = set(train_idx.tolist())
query_idx_set = set(query_idx.tolist())
retrieval_train_idx = np.array([i for i in range(50000) if i not in train_idx_set], dtype=np.int64)
retrieval_test_idx = np.array([i for i in range(10000) if i not in query_idx_set], dtype=np.int64)

print("Training:", len(train_idx))  # 5000
print("Query:", len(query_idx))     # 1000
print("mAP@54000 gallery:", len(retrieval_train_idx) + len(retrieval_test_idx)) # 54000
print("mAP@All gallery:", len(retrieval_train_idx) + len(retrieval_test_idx) + len(train_idx)) # 59000

100%|██████████| 170M/170M [1:06:30<00:00, 42.7kB/s]


Training: 5000
Query: 1000
mAP@54000 gallery: 54000
mAP@All gallery: 59000


In [11]:
class IndexedCIFAR(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = list(map(int, indices))
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        image, label = self.dataset[idx]
        return image, int(label)

class CombinedDataset(Dataset):
    def __init__(self, ds1, idx1, ds2, idx2):
        self.ds1, self.idx1 = ds1, list(map(int, idx1))
        self.ds2, self.idx2 = ds2, list(map(int, idx2))
    def __len__(self):
        return len(self.idx1) + len(self.idx2)
    def __getitem__(self, i):
        if i < len(self.idx1):
            x, y = self.ds1[self.idx1[i]]
        else:
            x, y = self.ds2[self.idx2[i - len(self.idx1)]]
        return x, int(y)

train_ds = IndexedCIFAR(train_aug, train_idx)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    drop_last=True
)

query_ds = IndexedCIFAR(test_eval, query_idx)
query_loader = DataLoader(query_ds, batch_size=128, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

# All 59k non-query images, used for mAP@All.
gallery_all_ds = CombinedDataset(
    train_eval, np.arange(50000),
    test_eval, retrieval_test_idx
)
gallery_all_loader = DataLoader(gallery_all_ds, batch_size=128, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

# 54k gallery excludes the 5k images used for training.
gallery_54k_ds = CombinedDataset(
    train_eval, retrieval_train_idx,
    test_eval, retrieval_test_idx
)
gallery_54k_loader = DataLoader(gallery_54k_ds, batch_size=128, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(len(train_ds), len(query_ds), len(gallery_54k_ds), len(gallery_all_ds))

5000 1000 54000 59000


In [5]:
weights = models.AlexNet_Weights.IMAGENET1K_V1
model = models.alexnet(weights=weights)
model.classifier[6] = nn.Linear(model.classifier[6].in_features, HASH_BITS)
model = model.to(device)

# Giữ backbone trainable, fine-tune toàn mạng.
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print("Model output bits:", HASH_BITS)

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 170MB/s]


Model output bits: 32


6. Pairwise weighted logistic loss + continuation


In [6]:
def hashnet_pairwise_loss(z, labels, beta):
    # Continuation: beta tăng dần để tanh(beta*z) gần sign hơn.
    u = torch.tanh(beta * z)
    theta = 0.5 * (u @ u.t())

    labels = labels.view(-1, 1)
    sim = (labels == labels.t()).float()
    n = sim.size(0)
    mask = ~torch.eye(n, dtype=torch.bool, device=sim.device)

    sim = sim[mask]
    theta = theta[mask]

    # Balance the total contribution of positive and negative pairs.
    pos_count = sim.sum().clamp_min(1.0)
    neg_count = (1.0 - sim).sum().clamp_min(1.0)
    eta = (pos_count / neg_count).detach()
    weights = torch.where(sim > 0.5, torch.ones_like(sim), eta)

    # BCEWithLogits: target 1 for similar, 0 for dissimilar.
    losses = F.binary_cross_entropy_with_logits(theta, sim, reduction="none")
    return (weights * losses).sum() / weights.sum().clamp_min(1e-12)

@torch.no_grad()
def collect_continuous_and_binary_codes(model, loader, beta=16):
    model.eval()
    continuous, binary, labels_all = [], [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        z = model(images)
        continuous.append(z.cpu())
        # Inference uses sign to form binary codes.
        binary.append(torch.sign(z).cpu())
        labels_all.append(labels.cpu())
    return (torch.cat(continuous).numpy(),
            torch.cat(binary).numpy(),
            torch.cat(labels_all).numpy())

7. Huấn luyện

In [7]:
history = []
start_time = time.time()

for beta in BETA_SCHEDULE:
    for epoch in range(1, EPOCHS_PER_BETA + 1):
        model.train()
        running_loss, n_batches = 0.0, 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            z = model(images)
            loss = hashnet_pairwise_loss(z, labels, beta)
            loss.backward()
            optimizer.step()

            running_loss += float(loss.detach().cpu())
            n_batches += 1

        avg_loss = running_loss / max(n_batches, 1)
        history.append({"beta": beta, "epoch": epoch, "loss": avg_loss})
        print(f"beta={beta:>2} | epoch={epoch:02d}/{EPOCHS_PER_BETA} | loss={avg_loss:.5f}")

    torch.save({
        "model_state_dict": model.state_dict(),
        "hash_bits": HASH_BITS,
        "beta_schedule": BETA_SCHEDULE,
        "epochs_per_beta": EPOCHS_PER_BETA,
        "seed": SEED,
    }, os.path.join(OUTPUT_DIR, f"hashnet_cifar10_{HASH_BITS}bit_checkpoint.pt"))

pd.DataFrame(history).to_csv(
    os.path.join(OUTPUT_DIR, f"training_history_{HASH_BITS}bit.csv"), index=False
)
print(f"Training time: {(time.time()-start_time)/60:.1f} minutes")

beta= 1 | epoch=01/20 | loss=0.60994
beta= 1 | epoch=02/20 | loss=0.53215
beta= 1 | epoch=03/20 | loss=0.50505
beta= 1 | epoch=04/20 | loss=0.47974
beta= 1 | epoch=05/20 | loss=0.46294
beta= 1 | epoch=06/20 | loss=0.44954
beta= 1 | epoch=07/20 | loss=0.43397
beta= 1 | epoch=08/20 | loss=0.42209
beta= 1 | epoch=09/20 | loss=0.40759
beta= 1 | epoch=10/20 | loss=0.39290
beta= 1 | epoch=11/20 | loss=0.38469
beta= 1 | epoch=12/20 | loss=0.37317
beta= 1 | epoch=13/20 | loss=0.36103
beta= 1 | epoch=14/20 | loss=0.34972
beta= 1 | epoch=15/20 | loss=0.33760
beta= 1 | epoch=16/20 | loss=0.32905
beta= 1 | epoch=17/20 | loss=0.31959
beta= 1 | epoch=18/20 | loss=0.31047
beta= 1 | epoch=19/20 | loss=0.30188
beta= 1 | epoch=20/20 | loss=0.29416
beta= 2 | epoch=01/20 | loss=0.28370
beta= 2 | epoch=02/20 | loss=0.26639
beta= 2 | epoch=03/20 | loss=0.25889
beta= 2 | epoch=04/20 | loss=0.24861
beta= 2 | epoch=05/20 | loss=0.24062
beta= 2 | epoch=06/20 | loss=0.22931
beta= 2 | epoch=07/20 | loss=0.22491
b

In [8]:
@torch.no_grad()
def extract_binary_codes(model, loader):
    model.eval()
    codes, labels = [], []
    for images, y in loader:
        images = images.to(device, non_blocking=True)
        z = model(images)
        # sign(0) is 0; replace zero with +1 for binary Hamming codes.
        b = torch.where(z >= 0, 1.0, -1.0)
        codes.append(b.cpu().numpy().astype(np.int8))
        labels.append(y.numpy())
    return np.concatenate(codes), np.concatenate(labels)

def mean_average_precision(query_codes, query_labels, db_codes, db_labels, topk=None):
    # Hamming distance for {-1,+1} codes: (K - q @ db.T)/2.
    q = query_codes.astype(np.int16)
    db = db_codes.astype(np.int16)
    bit_count = q.shape[1]
    distances = (bit_count - q @ db.T) // 2

    aps = []
    for i in range(len(query_labels)):
        order = np.argsort(distances[i], kind="stable")
        if topk is not None:
            order = order[:topk]
        relevant = (db_labels[order] == query_labels[i]).astype(np.int32)
        total_relevant = int(np.sum(db_labels == query_labels[i]))
        if topk is not None:
            denom = min(total_relevant, topk)
        else:
            denom = total_relevant
        if denom == 0:
            aps.append(0.0)
            continue
        precision_at_k = np.cumsum(relevant) / (np.arange(len(relevant)) + 1.0)
        aps.append(float(np.sum(precision_at_k * relevant) / denom))
    return float(np.mean(aps)), np.array(aps)

Đánh giá mAP@54000 và mAP@All

In [9]:
q_codes, q_labels = extract_binary_codes(model, query_loader)
db54_codes, db54_labels = extract_binary_codes(model, gallery_54k_loader)
dball_codes, dball_labels = extract_binary_codes(model, gallery_all_loader)

print("Shapes:", q_codes.shape, db54_codes.shape, dball_codes.shape)
assert len(q_labels) == 1000
assert len(db54_labels) == 54000
assert len(dball_labels) == 59000

map54k, ap54k = mean_average_precision(q_codes, q_labels, db54_codes, db54_labels)
map_all, ap_all = mean_average_precision(q_codes, q_labels, dball_codes, dball_labels)

results = pd.DataFrame([{
    "method": "HashNet-style (AlexNet, PyTorch)",
    "dataset": "CIFAR-10",
    "bits": HASH_BITS,
    "train_images": len(train_ds),
    "query_images": len(query_ds),
    "gallery_mAP@54000": len(gallery_54k_ds),
    "mAP@54000": map54k,
    "gallery_mAP@All": len(gallery_all_ds),
    "mAP@All": map_all,
    "beta_schedule": str(BETA_SCHEDULE),
    "epochs_per_beta": EPOCHS_PER_BETA,
    "seed": SEED,
}])
display(results)
results.to_csv(os.path.join(OUTPUT_DIR, f"results_{HASH_BITS}bit.csv"), index=False)
print("Saved results to:", OUTPUT_DIR)

Shapes: (1000, 32) (54000, 32) (59000, 32)


,method,dataset,bits,train_images,query_images,gallery_mAP@54000,mAP@54000,gallery_mAP@All,mAP@All,beta_schedule,epochs_per_beta,seed
0,"HashNet-style (AlexNet, PyTorch)",CIFAR-10,32,5000,1000,54000,0.793541,59000,0.803901,"[1, 2, 4, 8, 16]",20,3367


Saved results to: ./hashnet_outputs


In [10]:
SEED = 3367
HASH_BITS = 16          # thử 16, 32, 48, 64; mỗi lần chạy một giá trị
BATCH_SIZE = 64
EPOCHS_PER_BETA = 20      # 1–2 chỉ dùng để smoke-test, không phải thí nghiệm cuối
BETA_SCHEDULE = [1, 2, 4, 8, 16]
LR = 1e-5
NUM_WORKERS = 2
DATA_ROOT = "./data"
OUTPUT_DIR = "./hashnet_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [12]:
history = []
start_time = time.time()

for beta in BETA_SCHEDULE:
    for epoch in range(1, EPOCHS_PER_BETA + 1):
        model.train()
        running_loss, n_batches = 0.0, 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            z = model(images)
            loss = hashnet_pairwise_loss(z, labels, beta)
            loss.backward()
            optimizer.step()

            running_loss += float(loss.detach().cpu())
            n_batches += 1

        avg_loss = running_loss / max(n_batches, 1)
        history.append({"beta": beta, "epoch": epoch, "loss": avg_loss})
        print(f"beta={beta:>2} | epoch={epoch:02d}/{EPOCHS_PER_BETA} | loss={avg_loss:.5f}")

    torch.save({
        "model_state_dict": model.state_dict(),
        "hash_bits": HASH_BITS,
        "beta_schedule": BETA_SCHEDULE,
        "epochs_per_beta": EPOCHS_PER_BETA,
        "seed": SEED,
    }, os.path.join(OUTPUT_DIR, f"hashnet_cifar10_{HASH_BITS}bit_checkpoint.pt"))

pd.DataFrame(history).to_csv(
    os.path.join(OUTPUT_DIR, f"training_history_{HASH_BITS}bit.csv"), index=False
)
print(f"Training time: {(time.time()-start_time)/60:.1f} minutes")

beta= 1 | epoch=01/20 | loss=0.14692
beta= 1 | epoch=02/20 | loss=0.12537
beta= 1 | epoch=03/20 | loss=0.11676
beta= 1 | epoch=04/20 | loss=0.11230
beta= 1 | epoch=05/20 | loss=0.10887
beta= 1 | epoch=06/20 | loss=0.10603
beta= 1 | epoch=07/20 | loss=0.10585
beta= 1 | epoch=08/20 | loss=0.10371
beta= 1 | epoch=09/20 | loss=0.10286
beta= 1 | epoch=10/20 | loss=0.10190
beta= 1 | epoch=11/20 | loss=0.10079
beta= 1 | epoch=12/20 | loss=0.10022
beta= 1 | epoch=13/20 | loss=0.10005
beta= 1 | epoch=14/20 | loss=0.09918
beta= 1 | epoch=15/20 | loss=0.09848
beta= 1 | epoch=16/20 | loss=0.09728
beta= 1 | epoch=17/20 | loss=0.09750
beta= 1 | epoch=18/20 | loss=0.09629
beta= 1 | epoch=19/20 | loss=0.09675
beta= 1 | epoch=20/20 | loss=0.09505
beta= 2 | epoch=01/20 | loss=0.09429
beta= 2 | epoch=02/20 | loss=0.09371
beta= 2 | epoch=03/20 | loss=0.09326
beta= 2 | epoch=04/20 | loss=0.09261
beta= 2 | epoch=05/20 | loss=0.09234
beta= 2 | epoch=06/20 | loss=0.09214
beta= 2 | epoch=07/20 | loss=0.09239
b

In [13]:
q_codes, q_labels = extract_binary_codes(model, query_loader)
db54_codes, db54_labels = extract_binary_codes(model, gallery_54k_loader)
dball_codes, dball_labels = extract_binary_codes(model, gallery_all_loader)

print("Shapes:", q_codes.shape, db54_codes.shape, dball_codes.shape)
assert len(q_labels) == 1000
assert len(db54_labels) == 54000
assert len(dball_labels) == 59000

map54k, ap54k = mean_average_precision(q_codes, q_labels, db54_codes, db54_labels)
map_all, ap_all = mean_average_precision(q_codes, q_labels, dball_codes, dball_labels)

results = pd.DataFrame([{
    "method": "HashNet-style (AlexNet, PyTorch)",
    "dataset": "CIFAR-10",
    "bits": HASH_BITS,
    "train_images": len(train_ds),
    "query_images": len(query_ds),
    "gallery_mAP@54000": len(gallery_54k_ds),
    "mAP@54000": map54k,
    "gallery_mAP@All": len(gallery_all_ds),
    "mAP@All": map_all,
    "beta_schedule": str(BETA_SCHEDULE),
    "epochs_per_beta": EPOCHS_PER_BETA,
    "seed": SEED,
}])
display(results)
results.to_csv(os.path.join(OUTPUT_DIR, f"results_{HASH_BITS}bit.csv"), index=False)
print("Saved results to:", OUTPUT_DIR)

Shapes: (1000, 32) (54000, 32) (59000, 32)


,method,dataset,bits,train_images,query_images,gallery_mAP@54000,mAP@54000,gallery_mAP@All,mAP@All,beta_schedule,epochs_per_beta,seed
0,"HashNet-style (AlexNet, PyTorch)",CIFAR-10,16,5000,1000,54000,0.773362,59000,0.784326,"[1, 2, 4, 8, 16]",20,3367


Saved results to: ./hashnet_outputs
